# 🔄 通用多模态数据集转换脚本

高度模块化、分层抽象的一键全流程 notebook，适用于任意多模态评测数据集。

## 共通流程（全自动）
1. **L0 配置** — 用户填写数据集路径 & 元信息（唯一需要改的地方）
2. **L1 基础设施** — S3 上传 / URL 拼接 / 代理剥离（凭证固定）
3. **L2 媒体处理** — 图片上传S3、视频抽帧上传S3、远程图片下载→base64（数据集特有）
4. **L3 核心处理** — 读原始数据 → 构建 example → 分组写入 eval JSON → config CSV（数据集特有）
5. **L4 平台配置** — add_mm_output / operator 批量导入 JSON / 子集权重 JSON（**完全通用，无需改动**）
6. **L5 校验输出** — 验证 + 文件清单 + 后续操作指引（**完全通用**）

> **用户只需关注 Cell 1（配置）和 Cell 3（数据集特有处理逻辑），其余 Cell 从上到下依次运行即可。**


## 0. 安装依赖

In [ ]:
%pip install tqdm pandas boto3 botocore Pillow opencv-python-headless loguru requests


## 1. 📝 用户配置区（唯一需要改的地方）

修改下方路径和元信息，然后运行本 Cell。

| 字段 | 说明 |
|---|---|
| `DATASET_ROOT` | 原始数据集目录（图片/视频/JSON 都在这里） |
| `ORIGIN_DATA_PATH` | 原始数据 JSON 文件 |
| `OUTPUT_BASE` | 输出目录（eval_files / config / add_mm_output 都在这里） |
| `DATASET_NAME` | 平台 dataset_name |
| `BENCHMARK_NAME` | 平台 benchmark_name（须已在平台创建） |
| `OWNER` | 平台 owner |
| `SKIP_UPLOAD` | `False`=上传S3, `True`=只拼URL不上传 |


In [3]:
# ════════════════════════════════════════════════════════════════════════
# 📝 用户配置区 — 修改此处，其余 Cell 无需改动
# ════════════════════════════════════════════════════════════════════════

from pathlib import Path, PurePosixPath

# ── 数据集路径 ──
DATASET_ROOT   = Path("/mnt/dolphinfs/.../your_dataset_dir")
ORIGIN_DATA_PATH = DATASET_ROOT / "test.json"        # 或 train.json 等


# ── 平台元信息 ──
DATASET_NAME   = "YourDataset"          # 平台 dataset_name
BENCHMARK_NAME = "YourDataset"          # 平台 benchmark_name（须已创建）
OWNER          = "wb_liaoshihao"


# ── 输出目录 ──
OUTPUT_BASE = Path("/mnt/dolphinfs/ssd_pool/docker/user/hadoop-aipnlp/EVA/liaoshihao/processed_datasets") / Path(DATASET_NAME)

# ── 运行开关 ──
SKIP_UPLOAD = False   # False=上传S3, True=只拼URL
NOT_PING_URL = True   # True=跳过URL可达性校验（推荐）

# ── S3 常量（固定，无需改） ──
S3_HOST  = "s3plus-shon.meituan.net"
S3_BUCKET = "multimodal-eval"
S3_PREFIX = PurePosixPath(DATASET_NAME)

# ── 派生路径 ──
EVAL_DIR      = OUTPUT_BASE / "eval_files"
FRAME_DIR     = OUTPUT_BASE / "frames"       # 视频抽帧输出
ADD_MM_OUTPUT = OUTPUT_BASE / "add_mm_output"

for d in (OUTPUT_BASE, EVAL_DIR, FRAME_DIR, ADD_MM_OUTPUT):
    d.mkdir(parents=True, exist_ok=True)

print(f"原始数据: {ORIGIN_DATA_PATH}")
print(f"输出目录: {OUTPUT_BASE}")
print(f"S3 PREFIX: {S3_PREFIX}")
print("✅ 配置完成")


原始数据: /mnt/dolphinfs/.../your_dataset_dir/test.json
输出目录: /mnt/dolphinfs/ssd_pool/docker/user/hadoop-aipnlp/EVA/liaoshihao/processed_datasets/YourDataset
S3 PREFIX: YourDataset
✅ 配置完成


## 2. 🔧 基础设施层（S3 上传 + 媒体工具，通用无需改动）

包含 S3 上传、视频抽帧、远程图片下载→base64、URL 内网转换等工具函数。


In [ ]:
"""L1 基础设施层：S3 上传 + 媒体工具（通用）"""

import os, re, io, base64, time, traceback
from urllib.parse import unquote
from pathlib import PurePosixPath

import cv2
from tqdm import tqdm
from PIL import Image, ImageFile
import requests
from loguru import logger

try:
    import boto3
    from botocore.config import Config as BotoConfig
    from botocore.exceptions import ClientError as BotoClientError
    HAS_BOTO = True
except ImportError:
    HAS_BOTO = False
    BotoClientError = None

# ── S3 凭证（固定） ──
_S3_AK = "SRV_asxfwCx9Kv9zpm0PapTCUDPXZWhbH122"
_S3_SK = "U86fJaaF1TTSGWG5VJrZ1xGWQhTH6rN8"
_S3_EP = "http://mss-shon.vip.sankuai.com"

def _get_s3_client():
    return boto3.client("s3",
        aws_access_key_id=_S3_AK, aws_secret_access_key=_S3_SK,
        endpoint_url=_S3_EP,
        config=BotoConfig(signature_version="s3", connect_timeout=30, read_timeout=120),
        region_name="us-east-1")

def _s3_key_exists(key: str) -> bool:
    if not HAS_BOTO: return False
    hp, hs = os.environ.pop("http_proxy", None), os.environ.pop("https_proxy", None)
    try:
        _get_s3_client().head_object(Bucket=S3_BUCKET, Key=unquote(key)); return True
    except (BotoClientError, Exception): return False
    finally:
        if hp: os.environ["http_proxy"] = hp
        if hs: os.environ["https_proxy"] = hs

def upload_to_s3(local_path, s3_key: str) -> str:
    """上传本地文件到 S3，返回公网 URL。key 已存在则跳过。"""
    ck = unquote(s3_key)
    url = f"https://{S3_HOST}/{S3_BUCKET}/{ck}"
    if not HAS_BOTO: logger.warning("boto3 不可用，返回假定 URL"); return url
    if isinstance(local_path, Path): local_path = str(local_path)
    if not os.path.exists(local_path): logger.warning("文件不存在: {}", local_path); return url
    if _s3_key_exists(ck): return url
    hp, hs = os.environ.pop("http_proxy", None), os.environ.pop("https_proxy", None)
    try:
        _get_s3_client().upload_file(Filename=local_path, Bucket=S3_BUCKET, Key=ck,
                                     ExtraArgs={"ACL": "public-read"})
        logger.info("上传: {}", ck)
    except Exception as e: logger.error("上传失败 {}: {}", ck, e)
    finally:
        if hp: os.environ["http_proxy"] = hp
        if hs: os.environ["https_proxy"] = hs
    return url

def s3_url(local_path, s3_key: str, skip_upload: bool = False) -> str:
    """上传媒体到 S3 返回公网 URL；skip_upload=True 只拼 URL 不上传。"""
    ck = unquote(s3_key)
    if skip_upload: return f"https://{S3_HOST}/{S3_BUCKET}/{ck}"
    return upload_to_s3(local_path, s3_key)

# ── 视频抽帧 ──
_frame_cache = {}

def extract_frames(video_path, nframe: int = 8, frame_dir: Path = None) -> list:
    """均匀抽帧存 jpg，返回帧路径列表。带缓存 + 断点续传。"""
    if video_path in _frame_cache: return _frame_cache[video_path]
    if frame_dir is None: frame_dir = FRAME_DIR
    if isinstance(video_path, Path): video_path = str(video_path)
    vid_name = os.path.splitext(os.path.basename(video_path))[0]
    out_dir = frame_dir / vid_name
    out_dir.mkdir(parents=True, exist_ok=True)
    tag = f"n{nframe}"
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0: cap.release(); raise RuntimeError(f"cannot read: {video_path}")
    interval = max(total // nframe, 1)
    kept = [i for i in range(total) if i % interval == 0][:nframe]
    paths = [out_dir / f"frame_{tag}_{j:04d}.jpg" for j in range(len(kept))]
    if all(p.exists() for p in paths) and paths:
        cap.release(); _frame_cache[video_path] = paths; return paths
    kept_set = set(kept); pos = {idx: j for j, idx in enumerate(kept)}
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if i in kept_set:
            p = paths[pos[i]]
            if not p.exists(): cv2.imwrite(str(p), frame)
            if pos[i] == len(kept) - 1: break
        i += 1
    cap.release()
    if not all(p.exists() for p in paths): raise RuntimeError(f"抽帧不完整: {video_path}")
    _frame_cache[video_path] = paths
    return paths

# ── 远程图片下载 → base64 ──
def vip_url(url: str) -> str:
    """S3 外网 URL 转 VIP 内网地址"""
    return url.replace("s3plus-shon.meituan.net", "mss-shon.vip.sankuai.com").strip()

def download_image_to_memory(url: str) -> "Image | None":
    """从 URL 下载图片到内存，返回 PIL Image，不落盘。"""
    if not url: return None
    u = vip_url(url)
    hp, hs = os.environ.pop("http_proxy", None), os.environ.pop("https_proxy", None)
    resp = None
    for attempt in range(3):
        try:
            resp = requests.get(u, timeout=(30, 120)); resp.raise_for_status(); break
        except Exception as e:
            logger.warning("下载失败（第{}次） {}: {}", attempt+1, u, e)
            time.sleep(3)
    if hp: os.environ["http_proxy"] = hp
    if hs: os.environ["https_proxy"] = hs
    if resp is None or not resp.ok: return None
    try:
        ImageFile.LOAD_TRUNCATED_IMAGES = True
        img = Image.open(io.BytesIO(resp.content)); img.load()
        if img.mode in ("RGBA", "P", "LA"): img = img.convert("RGB")
        return img
    except Exception as e: logger.warning("图片解码失败 {}: {}", u, e); return None

def img2base64(img: "Image", fmt: str = "JPEG") -> str:
    """PIL Image → base64 字符串"""
    if img.mode in ("RGBA", "P", "LA"): img = img.convert("RGB")
    buf = io.BytesIO(); img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()

print("✅ 基础设施层就绪 (S3, 抽帧, 远程下载→base64)")


## 3. 🎯 数据集特有处理逻辑（需要修改）

这是唯一需要根据数据集格式修改的 Cell。你需要实现：

1. **`process_record(record)`** — 把一条原始数据记录转成平台 example dict:
   ```python
   {
     "id": ...,           # 唯一标识
     "input": ...,        # 带 <image_k>/<video_k> 占位符的 prompt
     "target": ...,       # 正确答案
     "extra_data": {
       "url": {...},      # {"<image_1>": "https://...S3_URL", ...}
       "origin_data": {},  # 原始记录（用于溯源）
       "openai-request": [{ "role": "user", "content": [{ "type": "image", "image": "https://..." }, { "type": "text", "text": "..." }] }]
     }
   }
   ```

2. **`get_subset_key(example)`** — 返回分组 key（用于按子集拆分 eval 文件）

下方提供 3 个参考模板（PhysBench / MemEye / UniSVG(TIR-Bench)），选一个修改即可。


In [ ]:
"""L2+L3: 数据集特有处理逻辑 — 修改此处适配你的数据集"""

import json, re
from collections import defaultdict
from pathlib import PurePosixPath

# ════════════════════════════════════════════════════════════════════════
# 🎯 选择/修改一个模板，取消注释即可使用
# ════════════════════════════════════════════════════════════════════════

# ─── 模板 A: PhysBench（视频抽帧 + 图片交错 + MCQ end_prompt）───
"""
PLACEHOLDER_RE = re.compile(r"<video>|<image>")
VIDEO_DESC = "The video is split to a series of images sampled at equal intervals from the beginning to the end of it, based on the series of images, answer the question."
END_PROMPT = "\nAnswer with the option's letter from the given choices directly. You can only answer one letter from A, B, C, or D."

def resolve_media_path(file_name, image_dir=None, video_dir=None):
    if image_dir is None: image_dir = DATASET_ROOT / "image"
    if video_dir is None: video_dir = DATASET_ROOT / "video"
    ext = file_name.rsplit(".", 1)[-1].lower()
    base_dir = video_dir if ext == "mp4" else image_dir
    p = base_dir / file_name
    if p.exists(): return p
    stem, _, e = file_name.rpartition(".")
    for cand in (f"{stem}.{e.upper()}", f"{stem}.{e.lower()}"):
        cp = base_dir / cand
        if cp.exists(): return cp
    return None

def process_record(record, answer_map):
    idx = record["idx"]
    question = record["question"]
    file_names = record["file_name"]
    answer = answer_map.get(idx)
    if answer is None: return None

    url = {}; content = []; input_parts = []; k = 0; file_ptr = 0; last = 0
    has_video = "<video>" in question
    if has_video: content.append({"type": "text", "text": VIDEO_DESC}); input_parts.append(VIDEO_DESC)

    for m in PLACEHOLDER_RE.finditer(question):
        pre = question[last:m.start()]
        if pre: content.append({"type": "text", "text": pre}); input_parts.append(pre)
        last = m.end()
        fn = file_names[file_ptr]; file_ptr += 1
        media_path = resolve_media_path(fn)
        if media_path is None: raise FileNotFoundError(f"idx={idx} media not found: {fn}")
        if m.group() == "<video>":
            frame_paths = extract_frames(str(media_path))
            vid_name = os.path.splitext(os.path.basename(str(media_path)))[0]
            ph = []
            for fp in frame_paths:
                k += 1; tag = f"<image_{k}>"
                s3k = (S3_PREFIX / "frames" / vid_name / os.path.basename(str(fp))).as_posix()
                s3u = s3_url(str(fp), s3k, skip_upload=SKIP_UPLOAD)
                url[tag] = s3u; content.append({"type": "image", "image": s3u}); ph.append(tag)
            input_parts.append("".join(ph))
        else:
            k += 1; tag = f"<image_{k}>"
            s3k = (S3_PREFIX / "image" / os.path.basename(str(media_path))).as_posix()
            s3u = s3_url(str(media_path), s3k, skip_upload=SKIP_UPLOAD)
            url[tag] = s3u; content.append({"type": "image", "image": s3u}); input_parts.append(tag)

    tail = question[last:] + END_PROMPT
    content.append({"type": "text", "text": tail}); input_parts.append(tail)
    messages = [{"role": "user", "content": content}]

    origin_data = {"idx": idx, "scene": record.get("scene"), "question": question,
                   "file_name": file_names, "split": record.get("split")}

    return {"id": str(idx), "input": "".join(input_parts), "target": answer,
            "extra_data": {"url": url, "origin_data": origin_data, "openai-request": messages},
            "_subset": record.get("split", "test")}

def get_subset_key(example): return example["_subset"]
"""

# ─── 模板 B: MemEye（多轮对话 + MCQ/Open + 图片S3上传）───
"""
PLACEHOLDER_RE = re.compile(r"<image(?:\s*_?\s*\d+)?>", re.IGNORECASE)

def process_record(record):
    # record 是一个已经处理好的 MemEye item dict
    # 这里简化为直接返回（实际需根据 dialog 结构构建多轮对话）
    # 参考 MemEye_transfer.ipynb 的 build_multi_turn_conversation()
    return record  # placeholder — 实际需自行实现

def get_subset_key(example): return example["_subset"]
"""

# ─── 模板 C: UniSVG/TIR-Bench（远程URL下载→base64 data URI）───
"""
def process_record(item):
    prompt = item.get("prompt", "")
    url = {}; content = []
    for key, field in ("<image_1>", "image_1"), ("<image_2>", "image_2"):
        if item.get(field):
            img = download_image_to_memory(item[field])
            if img:
                b64 = img2base64(img)
                data_uri = f"data:image/jpeg;base64,{b64}"
                url[key] = data_uri
                content.append({"type": "image", "image": data_uri})
    image_prefix = "".join(url.keys())
    prompt_text = f"{image_prefix}{prompt}"
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    origin_data = {"id": item.get("id"), "task": item.get("task"),
                   "answer": item.get("answer"), "prompt": prompt,
                   "image_1": item.get("image_1"), "image_2": item.get("image_2")}

    return {"id": str(item.get("id")), "input": prompt_text, "target": item.get("answer", ""),
            "extra_data": {"url": url, "origin_data": origin_data, "openai-request": messages},
            "_subset": item.get("task", "all")}

def get_subset_key(example): return example["_subset"]
"""

# ════════════════════════════════════════════════════════════════════════
# 🎯 在此实现你自己的 process_record 和 get_subset_key
# ════════════════════════════════════════════════════════════════════════

def process_record(record):
    """把一条原始数据记录转成平台 example dict。
    返回: {"id": ..., "input": ..., "target": ..., "extra_data": {...}, "_subset": ...}
    或 None 表示跳过该条。
    """
    # TODO: 实现你的逻辑
    raise NotImplementedError("请在 Cell 3 实现 process_record()")

def get_subset_key(example):
    """返回分组 key，用于按子集拆分 eval 文件。
    例如: "test", "val", "ISVGEN_SSIM", "Brand_Memory_Test_mcq" 等。
    """
    # TODO: 实现你的逻辑
    raise NotImplementedError("请在 Cell 3 实现 get_subset_key()")

print("✅ 数据集特有处理函数已定义（需实现 process_record / get_subset_key）")


## 4. ⚙️ 核心处理执行（通用，无需改动）

读原始数据 → 逐条调 `process_record` → 按 `get_subset_key` 分组 → 写 eval JSON + config CSV。


In [ ]:
"""L3 核心处理执行：读数据 → process_record → 分组 → eval JSON + config CSV"""

import json
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import pandas as pd

# ── 读取原始数据 ──
raw_data = json.loads(ORIGIN_DATA_PATH.read_text())
if isinstance(raw_data, dict) and "examples" in raw_data:
    raw_data = raw_data["examples"]  # 兼容已包装格式
print(f"原始数据: {len(raw_data)} 条")

# ── 逐条处理 ──
groups = defaultdict(list)
errors = []
skipped = 0

for idx, record in enumerate(tqdm(raw_data, desc="Processing")):
    try:
        example = process_record(record)
    except Exception as e:
        errors.append((idx, str(e))); continue
    if example is None:
        skipped += 1; continue
    subset = get_subset_key(example)
    # 清理内部标记（_subset 不写入 eval 文件）
    clean = {k: v for k, v in example.items() if not k.startswith("_")}
    groups[subset].append(clean)

print(f"成功: {sum(len(v) for v in groups.values())} 条, 跳过: {skipped}, 错误: {len(errors)}")
if errors:
    for idx, msg in errors[:10]:
        print(f"  [err] idx={idx}: {msg}")

# ── 写 eval JSON + config CSV ──
today = datetime.today().strftime("%m%d")
config_rows = []

for subset, examples in sorted(groups.items()):
    eval_path = EVAL_DIR / f"{DATASET_NAME}_{subset}.json"
    with eval_path.open("w") as f:
        json.dump({"examples": examples}, f, ensure_ascii=False, indent=4)
    print(f"  {subset}: {len(examples)} 条 -> {eval_path}")
    config_rows.append([
        DATASET_NAME, f"{DATASET_NAME}_{subset}", str(eval_path),
        "", f"{DATASET_NAME} - {subset}", 1,
    ])

config_path = OUTPUT_BASE / f"config_{today}.csv"
df_config = pd.DataFrame(config_rows, columns=[
    "dataset_name", "subset_name", "question_path", "image_path",
    "dataset_description", "preset",
])
df_config.to_csv(config_path, index=False)

print(f"\n✅ eval JSON + config CSV 完成")
print(f"  config: {config_path}")
print(f"  eval_files: {EVAL_DIR}/")


## 5. 🏗️ 平台配置层（通用，无需改动）

生成 add_mm_output 目录结构：
- `{DATASET}/{subset}/task_url.json` — 任务数据本体
- `config_benchmarks/{DATASET}.json` — benchmark 配置
- `run_specs_{DATASET}.conf` — 总体 run_specs
- `{ts}_add_{DATASET}.json` — operator 批量导入 JSON
- `subject_weight_{BENCHMARK}_{date}.json` — 子集权重

等价于 `video_mme_v2_transfer.ipynb` 步骤 3-4，完全通用。


In [ ]:
"""L4 平台配置层：add_mm_output / operator / weight（完全通用）"""

import json, os, sys
from pathlib import Path
from datetime import datetime
from urllib.parse import unquote
import pandas as pd

# ── 导入 TaskJsonCreator ──
WORKSPACE = Path("/mnt/dolphinfs/ssd_pool/docker/user/hadoop-aipnlp/EVA/liaoshihao")
sys.path.insert(0, str(WORKSPACE))
from utils.create_task import TaskJsonCreator

ADD_MM_OUTPUT.mkdir(parents=True, exist_ok=True)

entries_list, all_bad_urls = [], []

# ── 步骤 3a: TaskJsonCreator ──
for ds_name, df_group in df_config.groupby("dataset_name"):
    desc = df_group.iloc[0]["dataset_description"]
    creator = TaskJsonCreator(output_path=str(ADD_MM_OUTPUT), dataset_name=ds_name, description=desc)
    print(f"\n=== {ds_name} (output: {creator.get_dataset_root_path()}) ===")

    for _, row in df_group.iterrows():
        subset = row["subset_name"]
        preset = int(row.get("preset", 0))
        if preset != 1:
            print(f"  跳过非 preset: {subset}"); continue
        examples = json.loads(Path(row["question_path"]).read_text()).get("examples", [])
        task_path, bad_urls = creator.create_copy_task(examples, subset_name=subset, ping_url=not NOT_PING_URL)
        if bad_urls: all_bad_urls.extend(bad_urls)
        print(f"  -> {task_path} ({len(examples)} 条)")

    cfg_json, conf_run, entries = creator.create_run_specs_conf()
    entries_list.extend(entries)
    print(f"  config:    {cfg_json}")
    print(f"  run_specs: {conf_run}")

# ── 总体 run_specs ──
overall_conf = ADD_MM_OUTPUT / f"run_specs_{DATASET_NAME}.conf"
overall_conf.write_text("entries: [\n" + "".join(entries_list) + "]\n")
print(f"\n总体 run_specs: {overall_conf}")

# ── operator 批量导入 JSON ──
operator_items = []
config_benchmarks = ADD_MM_OUTPUT / "config_benchmarks"
for ds_name in df_config["dataset_name"].unique():
    cfg_file = config_benchmarks / f"{ds_name}.json"
    if not cfg_file.exists(): print(f"  跳过: {cfg_file} 不存在"); continue
    cfg = json.loads(cfg_file.read_text())
    for subset, task_path in cfg.get("data", {}).items():
        ds_path = f"{ds_name}:{subset}"
        desc = f"general_multimodal:benchmark={ds_name},adaptor=generation_en,subset={subset},model=###MODEL-NAME###"
        operator_items.append({
            "entity": "modelEvalDataSubSet", "action": "set",
            "data": {
                "type": "OFFICIAL", "statSourceType": "instance", "instanceType": "8",
                "execType": "CUSTOM", "dataSubSetName": ds_path, "dataSubSetLabel": ds_path,
                "description": ds_path, "statType": "COMMON",
                "autoEvalDataSizeOrderList": ["200", "50", "20", "5"],
                "publicStatus": "WHITE", "categoryType": "LLM-EVAL",
                "runtimeConfig": "{}",
                "tags": [{"name": "DATA_SUB_SET_SOURCE", "values": ["私有数据"]}],
                "categoryPathList": [{"metaVersionName": "类目体系V2",
                                    "path": "类目体系->能力项->交互能力->图像理解"}],
                "files": [{"name": "task_url.json", "detail": {
                    "bucket": "bucket", "index": "index", "path": task_path,
                    "dataSetName": ds_name, "dataSetVersionName": ds_name},
                    "sourceType": "dolphinfs", "dataKey": "test"}],
                "bindingRunSpecList": [{"description": desc,
                    "extraParam": '{"priority": 1, "overwrite": false }',
                    "statNameList": ["quasi_prefix_exact_match", "pass_at_k_by_platform"]}],
                "ownerList": [OWNER], "evalType": "auto",
            },
        })

now_ts = datetime.today().strftime("%Y%m%d%H")
operator_file = ADD_MM_OUTPUT / f"{now_ts}_add_{DATASET_NAME}.json"
operator_file.write_text(json.dumps(operator_items, ensure_ascii=False, indent=4))
print(f"\noperator 批量配置: {operator_file}  ({len(operator_items)} items)")
if all_bad_urls: print(f"[WARN] {len(all_bad_urls)} 个坏 URL")

# ── 步骤 4: 子集权重 ──
today_md = datetime.today().strftime("%m%d")
for idx, row in df_config.iterrows():
    try:
        data = json.loads(Path(row["question_path"]).read_text())
        df_config.at[idx, "num_questions"] = len(data.get("examples", []))
    except Exception as e:
        print(f"读取失败 {row['question_path']}: {e}")
        df_config.at[idx, "num_questions"] = 0

weight_item = {"entity": "modelEvalRunSpecSetDataSubSet", "action": "set",
               "data": {"name": BENCHMARK_NAME, "dataSubSetList": []}}
for _, row in df_config.iterrows():
    path = f"{row['dataset_name']}:{row['subset_name']}"
    weight_item["data"]["dataSubSetList"].append({"path": path,
        "weight": int(row.get("num_questions", 0) or 0)})

weight_path = OUTPUT_BASE / f"subject_weight_{BENCHMARK_NAME}_{today_md}.json"
weight_path.write_text(json.dumps([weight_item], ensure_ascii=False, indent=4))
print(f"\n权重配置: {weight_path}")

print("\n✅ add_mm_output 完成")


## 6. ✅ 校验 + 文件清单 + 后续指引


In [ ]:
"""L5 校验 + 文件清单"""

import os, json
from pathlib import Path

# ── 校验 eval 文件中 URL 来源 ──
local_count = 0; s3_count = 0; b64_count = 0
for p in sorted(EVAL_DIR.glob("*.json")):
    data = json.loads(p.read_text())
    for ex in data.get("examples", []):
        for k, url in ex.get("extra_data", {}).get("url", {}).items():
            if "meituan.net" in url or "sankuai.com" in url: s3_count += 1
            elif url.startswith("data:"): b64_count += 1
            else: local_count += 1

print(f"URL 类型统计: S3={s3_count}, base64={b64_count}, 本地={local_count}")
if local_count > 0:
    print("\n⚠️  还有本地路径，需确保后续平台可访问")
else:
    print("\n✅ 所有媒体 URL 均为 S3/base64 地址")

# ── 输出文件清单 ──
print("\n=== 生成的全部文件 ===")
for p in sorted(OUTPUT_BASE.rglob("*")):
    if p.is_file():
        sz = p.stat().st_size
        rel = p.relative_to(OUTPUT_BASE)
        print(f"  {sz:>14,} bytes  {rel}")

# ── 后续步骤 ──
print("\n=== 后续手动操作 ===")
print("1. 新增子集: 拿 operator 批量配置 JSON 去[平台](https://model.sankuai.com/admin/modelEval/19/dataBatch)批量导入")
print("2. 设置权重: 拿 subject_weight JSON 提交到同一页面（前提: benchmark 已创建）")
print("3. task 数据本体在 add_mm_output/{DATASET}/{subset}/task_url.json")
print("\n✅ 全部完成")
